# AI in Healthcare: Twitter Topic Analysis with SocialAU

## Scenario

We simulate a Twitter discussion on *"AI in healthcare"* involving:

| Layer | Size | Description |
|---|---|---|
| Users | 10 | Mix of influencers and regular accounts |
| Items | 8 | Articles/papers (clinical AI vs drug discovery) |
| Keywords | 12 | Medical terms (0–5) and ML terms (6–11) |

**Two influencers** (users 0 and 1) have many followers but post moderately.
**User 2** is highly active (most interactions in the tensor) but has no followers.

## Learning objectives

1. Understand how SocialAU combines network authority with topic activity.
2. See why TOPHITS (pure tensor) and SocialAU produce different rankings.
3. Observe the role of graph layers in surfacing *structurally* important users.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from influencer import edge2adj, socialAU, tophits

torch.manual_seed(42)

# User network (10x10): users 0 and 1 are influencers with many followers
mu_edges = [
    [2,0],[3,0],[4,0],[5,0],[6,0],   # 5 users follow influencer 0
    [7,1],[8,1],[9,1],[3,1],          # 4 users follow influencer 1
]
MU = torch.tensor(edge2adj(mu_edges, dim=10), dtype=torch.float32)

# Item network (8x8): two sub-topic clusters — clinical AI (0-3) and drug discovery (4-7)
mi_edges = [
    [0,1],[1,0],[0,2],[2,0],[1,2],[2,1],[2,3],[3,2],[1,3],[3,1],
    [4,5],[5,4],[4,6],[6,4],[5,6],[6,5],[5,7],[7,5],[6,7],[7,6],
]
MI = torch.tensor(edge2adj(mi_edges, dim=8), dtype=torch.float32)

# Keyword network (12x12): medical terms (0-5) and ML terms (6-11) form two co-occurrence clusters
mk_edges = [
    [0,1],[1,0],[0,2],[2,0],[1,2],[2,1],[2,3],[3,2],[3,4],[4,3],[4,5],[5,4],
    [6,7],[7,6],[6,8],[8,6],[7,8],[8,7],[8,9],[9,8],[9,10],[10,9],[10,11],[11,10],
]
MK = torch.tensor(edge2adj(mk_edges, dim=12), dtype=torch.float32)

# Tensor T (10x8x12): user-item-keyword interaction counts
T = torch.zeros(10, 8, 12)
T[0,0,0]=2; T[0,0,1]=2; T[0,1,0]=2           # influencer 0: clinical AI + medical terms
T[1,4,6]=2; T[1,5,7]=2; T[1,4,7]=2           # influencer 1: drug discovery + ML terms
T[2,0,0]=6; T[2,0,1]=5; T[2,1,0]=4; T[2,1,1]=4; T[2,0,2]=3  # user 2: very active, no followers
T[3,2,3]=1; T[4,0,0]=1; T[5,1,1]=1
T[6,3,2]=1; T[7,4,6]=1; T[8,5,7]=1; T[9,6,8]=1

print(f"Tensor T shape       : {list(T.shape)}")
print(f"Non-zero entries in T: {T.nonzero().shape[0]}")
print(f"User 2 total activity: {T[2].sum().item():.0f}  (most active)")
print(f"User 0 total activity: {T[0].sum().item():.0f}  (5 followers)")
print(f"User 1 total activity: {T[1].sum().item():.0f}  (4 followers)")

In [ ]:
# Run SocialAU: combines user/item/keyword graphs with tensor interactions
h_raw, a_raw, w_raw = socialAU(MU, MI, MK, T, epsilon=1e-6)
h_sau = h_raw.squeeze(0)   # user scores  (10,)
a_sau = a_raw.squeeze(0)   # item scores  (8,)
w_sau = w_raw.squeeze(0)   # keyword scores (12,)

# Run TOPHITS: pure tensor decomposition — no graph layer information
u_th, v_th, w_th = tophits(T)

print("Both algorithms converged.")
print(f"SocialAU output shapes — users: {h_sau.shape}, items: {a_sau.shape}, keywords: {w_sau.shape}")

In [ ]:
def top5_table(scores, label, entity="User"):
    ranked = sorted(enumerate(scores.tolist()), key=lambda x: x[1], reverse=True)[:5]
    print(f"Top-5 {entity}s — {label}")
    print(f"  {'Rank':>4} | {entity:>6} | {'Score':>8}")
    print(f"  {'----':>4} | {'------':>6} | {'--------':>8}")
    for rank, (idx, score) in enumerate(ranked, 1):
        print(f"  {rank:>4} | {idx:>6} | {score:>8.4f}")
    print()

top5_table(h_sau, "SocialAU")
top5_table(u_th,  "TOPHITS")

### What the comparison shows

| User | Followers | Tensor activity | TOPHITS rank | SocialAU rank |
|------|-----------|-----------------|--------------|---------------|
| 2    | 0         | 22 (highest)    | **1st**      | 1st (closer)  |
| 0    | 5         | 6               | 2nd          | **2nd (up)**  |
| 1    | 4         | 6               | not in top-5 | **3rd (up)**  |

TOPHITS ranks purely by interaction mass: user 2 dominates with a score of ~0.95.
SocialAU narrows that lead to ~0.77 while lifting users 0 and 1 (scores ~0.56 and ~0.25).
The graph layers add an **authority bonus** — users with many followers receive higher
`h_U + a_U` contributions to their final score, partially offsetting a lower tensor mass.

In [ ]:
sau_ranked = sorted(enumerate(h_sau.tolist()), key=lambda x: x[1], reverse=True)[:5]
th_ranked  = sorted(enumerate(u_th.tolist()),  key=lambda x: x[1], reverse=True)[:5]

sau_labels = [f"User {i}" for i, _ in sau_ranked]
sau_scores = [s for _, s in sau_ranked]
th_labels  = [f"User {i}" for i, _ in th_ranked]
th_scores  = [s for _, s in th_ranked]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.barh(sau_labels[::-1], sau_scores[::-1], color="steelblue")
ax1.set_xlabel("SocialAU Score")
ax1.set_title("Top-5 Users — SocialAU")
ax1.set_xlim(0, 1.05)

ax2.barh(th_labels[::-1], th_scores[::-1], color="coral")
ax2.set_xlabel("TOPHITS Score")
ax2.set_title("Top-5 Users — TOPHITS")
ax2.set_xlim(0, 1.05)

plt.suptitle("AI in Healthcare: Influence Rankings Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Interpretation

### Why the rankings differ

TOPHITS performs a rank-1 PARAFAC decomposition of the raw interaction tensor T.
It finds the combination (user, item, keyword) that explains the most variance — which
means it rewards raw *posting volume*. User 2, with 22 interaction units spread across
5 cells, naturally dominates the first PARAFAC component.

SocialAU adds three adjacency matrices that encode *who endorses whom*, *which items
are related*, and *which keywords co-occur*. At each iteration, each user's score
receives an additive contribution from the HITS authority score computed on the user
graph (`h_U + a_U`). Users 0 and 1 are pointed to by 5 and 4 followers respectively,
giving them high HITS authority and a substantial boost that pure tensor activity cannot
provide.

### Connection to the paper's goal

The SocialAU paper targets users that are **"both authoritative in their own network
AND active on the topic"**. This is exactly the tension this notebook demonstrates:

- **TOPHITS finds activity** — useful for trending-topic detection.
- **SocialAU finds authority-weighted activity** — more useful for identifying
  influencers whose posts will propagate widely because their followers amplify them.

In a real deployment, user 2's high raw activity might be noise (a bot, a spam account)
while users 0 and 1, though less prolific, carry genuine social influence.